# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 07. Destination-Side Robustness and Statistical-Power Diagnostics

This notebook tests whether the preliminary H1a destination-side findings are inconclusive because of limited precision, a particular spatial selection rule, the commuter-shock definition, influential MSOAs, or residual spatial dependence.

It does **not** search for a specification that produces statistical significance. The pre-specified 500 m cumulative-shock model from Notebook 07 remains the main model. Alternative specifications are used only to assess stability.

The tests are:

1. coefficient-scale minimum detectable effects;
2. 300 m, 500 m and 800 m office-submarket inclusion rules;
3. cumulative, Monday, Friday and year-specific shock definitions;
4. leave-one-MSOA-out influence analysis;
5. residual Moran's I using four-nearest-neighbour spatial weights.

In [ ]:
from pathlib import Path
import os
from difflib import get_close_matches
import re
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.spatial import distance_matrix
from scipy.stats import norm, t
from shapely import wkb
import statsmodels.api as sm
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

def find_repository_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within a cloned copy of the repository.")

REPO_ROOT = find_repository_root()
BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", REPO_ROOT)).expanduser().resolve()
PUBLIC_DATA_DIR = REPO_ROOT / "data" / "public"
PUBLIC_BOUNDARY_DIR = PUBLIC_DATA_DIR / "boundaries"
PUBLIC_ONS_DIR = PUBLIC_DATA_DIR / "ons"
PUBLIC_OPENLOCAL_DIR = PUBLIC_DATA_DIR / "openlocal"
DERIVED_DATA_DIR = REPO_ROOT / "data" / "derived"
SOURCE_DIR = BASE / "outputs" / "restricted_h1_fine_grained_analysis"
MSOA_DIR = BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
OUT_DIR = BASE / "outputs" / "restricted_h1_destination_robustness"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "orange": "#d88958", "orange_dark": "#a65432", "blue": "#5f9fc7",
    "blue_dark": "#2d6f9f", "green": "#6f9f8f", "ink": "#263238",
    "grey": "#8d9497", "light": "#f2f1ed", "red": "#b45f55",
}
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9})

files = {
    "station_shock": DERIVED_DATA_DIR / "tfl_all_station_commuter_shock.csv",
    "main_models": SOURCE_DIR / "h1_destination_comparison_models.csv",
    "station_points": PUBLIC_BOUNDARY_DIR / "Underground_Stations.geojson",
    "office_markets": PUBLIC_BOUNDARY_DIR / "London_Office_Markets_V1.geojson",
    "msoa_boundaries": PUBLIC_BOUNDARY_DIR / "london_msoa_2021_boundaries.geojson",
    "openlocal": Path(os.environ.get(
        "OPENLOCAL_PROPERTY_FILE",
        REPO_ROOT / "data" / "external" / "openlocal" / "openlocal_property_level.parquet",
    )),
}
missing = [name for name, path in files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")

## 1. Rebuild the Office-Submarket Station Universe

In [ ]:
def clean_station_name(value):
    value = "" if pd.isna(value) else str(value).strip()
    for suffix in [" LU", "(LU)", " LO", " DLR", " NR"]:
        value = value.replace(suffix, "")
    return re.sub(r"\s+", " ", value.replace(" & ", " and ").replace("St. ", "St ")).strip()

def match_key(value):
    value = clean_station_name(value).lower()
    value = re.sub(r"\([^)]*\)", "", value).replace("'", "").replace(".", "")
    value = re.sub(r"\bstation\b", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    return {"bank and monument": "bank"}.get(value, value)

shock = pd.read_csv(files["station_shock"])
shock["match_key"] = shock["clean_name"].map(match_key)
shock["shock_cumulative"] = shock["cumulative_collapse_score"]
shock["shock_monday"] = shock[["collapse_23_mon", "collapse_24_mon", "collapse_25_mon"]].sum(axis=1)
shock["shock_friday"] = shock[["collapse_23_fri", "collapse_24_fri", "collapse_25_fri"]].sum(axis=1)
shock["shock_2023"] = shock["total_collapse_23"]
shock["shock_2024"] = shock["total_collapse_24"]
shock["shock_2025"] = shock["total_collapse_25"]

stations = gpd.read_file(files["station_points"]).to_crs("EPSG:27700")
stations["match_key"] = stations["NAME"].map(match_key)
stations = stations.dissolve("match_key", as_index=False)
all_stations = stations.merge(shock, on="match_key", how="right")
all_stations = gpd.GeoDataFrame(all_stations, geometry="geometry", crs="EPSG:27700")

available = stations["match_key"].dropna().tolist()
unmatched = all_stations.loc[all_stations.geometry.isna(), "match_key"].dropna().unique()
fuzzy = {}
for key in unmatched:
    candidate = get_close_matches(key, available, n=1, cutoff=0.86)
    if candidate:
        fuzzy[key] = candidate[0]
if fuzzy:
    station_geometry = stations.set_index("match_key").geometry
    mask = all_stations.geometry.isna() & all_stations["match_key"].isin(fuzzy)
    all_stations.loc[mask, "geometry"] = all_stations.loc[mask, "match_key"].map(fuzzy).map(station_geometry)

crosswalk = {
    "West End": ["Mayfair", "Soho", "St James's", "Covent Garden", "Fitzrovia", "North of Oxford Street", "Paddington", "Knightsbridge", "Victoria"],
    "City": ["City Core"],
    "Tech Belt & Midtown": ["Midtown", "Bloomsbury", "Clerkenwell", "Euston", "Kings Cross", "Shoreditch", "Camden", "Aldgate & Whitechapel"],
    "Canary Wharf": ["Canary Wharf"],
    "Southbank": ["Southbank", "Waterloo", "Vauxhall", "Nine Elms and Battersea"],
}
market_to_group = {market: group for group, markets in crosswalk.items() for market in markets}
office = gpd.read_file(files["office_markets"]).to_crs("EPSG:27700")
office["study_submarket"] = office["Market"].map(market_to_group)
core_office = office[office["study_submarket"].notna()].copy()
msoa = gpd.read_file(files["msoa_boundaries"]).to_crs("EPSG:27700")

located = all_stations.dropna(subset=["geometry"]).copy()
located = gpd.sjoin_nearest(
    located, core_office[["Market", "study_submarket", "geometry"]],
    how="left", distance_col="distance_to_submarket_m",
).drop(columns="index_right", errors="ignore")
located = located.sort_values(["clean_name", "distance_to_submarket_m"]).drop_duplicates("clean_name")

shock_columns = ["shock_cumulative", "shock_monday", "shock_friday", "shock_2023", "shock_2024", "shock_2025"]

def modal_value(series):
    mode = series.dropna().mode()
    return mode.iloc[0] if len(mode) else np.nan

def weighted_average(group, column):
    valid = group[column].notna() & group["2019_Midweek"].fillna(0).gt(0)
    if valid.any():
        return np.average(group.loc[valid, column], weights=group.loc[valid, "2019_Midweek"])
    return group[column].mean()

def build_target(radius):
    pool = located[located["distance_to_submarket_m"].le(radius)].copy()
    pool = gpd.sjoin(pool, msoa[["MSOA21CD", "MSOA21NM", "geometry"]], how="inner", predicate="within").drop(columns="index_right", errors="ignore")
    rows = []
    for (code_value, name), group in pool.groupby(["MSOA21CD", "MSOA21NM"]):
        row = {
            "MSOA21CD": code_value, "MSOA21NM": name,
            "station_count": group["clean_name"].nunique(),
            "top100_station_count": int(group["is_top100"].sum()),
            "study_submarket": modal_value(group["study_submarket"]),
        }
        for column in shock_columns:
            row[column] = weighted_average(group, column)
        rows.append(row)
    target = pd.DataFrame(rows)
    target["high_affected_group"] = target["top100_station_count"].gt(0).astype(int)
    target["radius_m"] = radius
    return target, pool

targets = {}
pools = {}
for radius in [300, 500, 800]:
    targets[radius], pools[radius] = build_target(radius)

sample_summary = pd.concat([
    target.groupby("high_affected_group").agg(
        msoas=("MSOA21CD", "nunique"), stations=("station_count", "sum"),
        mean_shock=("shock_cumulative", "mean"), sd_shock=("shock_cumulative", "std"),
    ).reset_index().assign(radius_m=radius)
    for radius, target in targets.items()
], ignore_index=True)
sample_summary["group"] = sample_summary["high_affected_group"].map({0: "Less-affected", 1: "Top-100 affected"})
sample_summary.to_csv(OUT_DIR / "buffer_sample_summary.csv", index=False)
display(sample_summary[["radius_m", "group", "msoas", "stations", "mean_shock", "sd_shock"]])

## 2. Construct a Common OpenLocal Outcome Panel

In [ ]:
target_years = [2019, 2023, 2024, 2025]
target_periods = [f"{year}-01-01" for year in target_years]
all_target_codes = set().union(*[set(target["MSOA21CD"]) for target in targets.values()])

ol_columns = ["period", "uarn", "total_floor_area", "rateable_value", "geometry", "category_group"]
ol = pd.read_parquet(
    files["openlocal"], columns=ol_columns,
    filters=[("category_group", "==", "RETAIL"), ("period", "in", target_periods)],
)
ol["year"] = pd.to_datetime(ol["period"]).dt.year
for column in ["total_floor_area", "rateable_value"]:
    ol[column] = pd.to_numeric(ol[column], errors="coerce")

def parse_wkb(value):
    try:
        return wkb.loads(bytes.fromhex(str(value)))
    except Exception:
        return None

ol["geometry"] = ol["geometry"].map(parse_wkb)
ol = ol.dropna(subset=["geometry", "uarn"])
ol_points = gpd.GeoDataFrame(ol, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:27700")
ol_msoa = gpd.sjoin(ol_points, msoa[["MSOA21CD", "MSOA21NM", "geometry"]], how="inner", predicate="within")
ol_msoa = ol_msoa[ol_msoa["MSOA21CD"].isin(all_target_codes)].copy()
ol_year_all = ol_msoa.groupby(["MSOA21CD", "MSOA21NM", "year"], as_index=False).agg(
    retail_units=("uarn", "nunique"),
    total_rateable_value=("rateable_value", "sum"),
    total_floor_area=("total_floor_area", "sum"),
)

metrics = ["retail_units", "total_rateable_value", "total_floor_area"]

def build_panel(target):
    year = ol_year_all[ol_year_all["MSOA21CD"].isin(target["MSOA21CD"])].merge(
        target, on=["MSOA21CD", "MSOA21NM"], how="inner"
    )
    baseline = year[year["year"].eq(2019)][["MSOA21CD"] + metrics].rename(
        columns={metric: f"{metric}_2019" for metric in metrics}
    )
    panel = year[year["year"].isin([2023, 2024, 2025])].merge(baseline, on="MSOA21CD", how="inner")
    for metric in metrics:
        panel[f"{metric}_log_change_2019"] = np.log1p(panel[metric].clip(lower=0)) - np.log1p(panel[f"{metric}_2019"].clip(lower=0))
        panel[f"log_{metric}_2019"] = np.log1p(panel[f"{metric}_2019"].clip(lower=0))
    return panel

panels = {radius: build_panel(target) for radius, target in targets.items()}
print({radius: (panel["MSOA21CD"].nunique(), len(panel)) for radius, panel in panels.items()})

## 3. Minimum Detectable Effects

The minimum detectable effect (MDE) is the smallest coefficient that the current design could distinguish from zero with a chosen level of statistical power, conditional on the observed clustered standard error. It is a precision diagnostic, not proof that the true effect has that size.

A large MDE relative to the estimated coefficient means that the design can rule out only large effects. In that situation, a non-significant result should be described as imprecise rather than as evidence of no relationship.

In [ ]:
main_models = pd.read_csv(files["main_models"])
power_rows = main_models[(main_models["source"].eq("OpenLocal")) & (main_models["exposure_label"].eq("Continuous shock"))].copy()
power_rows["df"] = power_rows["n_msoas"] - 1
power_rows["mde_80"] = (t.ppf(0.975, power_rows["df"]) + norm.ppf(0.80)) * power_rows["clustered_se"]
power_rows["mde_90"] = (t.ppf(0.975, power_rows["df"]) + norm.ppf(0.90)) * power_rows["clustered_se"]
power_rows["estimate_to_mde80"] = power_rows["beta"].abs() / power_rows["mde_80"]
power_table = power_rows[["outcome_label", "n_msoas", "beta", "clustered_se", "mde_80", "mde_90", "estimate_to_mde80"]]
power_table.to_csv(OUT_DIR / "minimum_detectable_effects.csv", index=False)

fig, ax = plt.subplots(figsize=(8.3, 3.4))
y = np.arange(len(power_table))
ax.barh(y + 0.17, power_table["mde_80"], height=0.28, color=PALETTE["blue"], label="80% MDE")
ax.barh(y - 0.17, power_table["beta"].abs(), height=0.28, color=PALETTE["orange"], label="Absolute estimate")
ax.set_yticks(y, power_table["outcome_label"])
ax.set_xlabel("Absolute standardized-shock coefficient")
ax.set_title("Current H1a Precision: Estimated Effects and 80% Minimum Detectable Effects", loc="left")
ax.legend(frameon=False, ncol=2)
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_01_minimum_detectable_effects.png", bbox_inches="tight")
plt.show()
display(power_table.round(4))

## 4. Shared Model Helper

In [ ]:
outcome_specs = {
    "Retail unit count": ("retail_units_log_change_2019", "log_retail_units_2019"),
    "Total rateable value": ("total_rateable_value_log_change_2019", "log_total_rateable_value_2019"),
    "Retail floor area": ("total_floor_area_log_change_2019", "log_total_floor_area_2019"),
}

def fit_destination(data, outcome, baseline, exposure, standardize=True, return_residuals=False):
    columns = ["MSOA21CD", "year", "study_submarket", outcome, baseline, exposure]
    clean = data[columns].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(clean) < 20 or clean["MSOA21CD"].nunique() < 8 or clean[exposure].nunique() < 2:
        return None
    if standardize:
        sd = clean[exposure].std(ddof=0)
        clean["exposure_model"] = (clean[exposure] - clean[exposure].mean()) / sd
    else:
        clean["exposure_model"] = clean[exposure].astype(float)
    year_dummies = pd.get_dummies(clean["year"].astype(str), prefix="year", drop_first=True, dtype=float)
    market_dummies = pd.get_dummies(clean["study_submarket"].astype(str), prefix="market", drop_first=True, dtype=float)
    X = pd.concat([
        clean[["exposure_model", baseline]].astype(float).reset_index(drop=True),
        year_dummies.reset_index(drop=True), market_dummies.reset_index(drop=True),
    ], axis=1)
    X = sm.add_constant(X)
    y = clean[outcome].astype(float).reset_index(drop=True)
    groups = clean["MSOA21CD"].reset_index(drop=True)
    fitted = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": groups})
    result = {
        "n_obs": int(fitted.nobs), "n_msoas": int(clean["MSOA21CD"].nunique()),
        "beta": fitted.params["exposure_model"], "se": fitted.bse["exposure_model"],
        "p_value": fitted.pvalues["exposure_model"], "r_squared": fitted.rsquared,
    }
    if return_residuals:
        residuals = clean[["MSOA21CD"]].reset_index(drop=True).copy()
        residuals["residual"] = fitted.resid
        result["residuals"] = residuals
    return result

## 5. Buffer Sensitivity

The 500 m rule is the pre-specified main definition. The 300 m and 800 m versions test whether the results depend on an unusually narrow or broad interpretation of station proximity to an office submarket. Stable coefficient directions are more important than isolated p-values.

In [ ]:
buffer_rows = []
for radius, panel in panels.items():
    for outcome_label, (outcome, baseline) in outcome_specs.items():
        for exposure, label, standardize in [
            ("shock_cumulative", "Continuous shock", True),
            ("high_affected_group", "Top-100 group", False),
        ]:
            result = fit_destination(panel, outcome, baseline, exposure, standardize=standardize)
            if result:
                result.update({"radius_m": radius, "outcome": outcome_label, "exposure": label})
                buffer_rows.append(result)
buffer_results = pd.DataFrame(buffer_rows)
buffer_results["ci_low"] = buffer_results["beta"] - 1.96 * buffer_results["se"]
buffer_results["ci_high"] = buffer_results["beta"] + 1.96 * buffer_results["se"]
buffer_results.to_csv(OUT_DIR / "buffer_sensitivity_models.csv", index=False)

continuous = buffer_results[buffer_results["exposure"].eq("Continuous shock")]
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))
colors = {300: PALETTE["green"], 500: PALETTE["orange_dark"], 800: PALETTE["blue_dark"]}
for ax, outcome_label in zip(axes, outcome_specs):
    subset = continuous[continuous["outcome"].eq(outcome_label)]
    for y, radius in enumerate([300, 500, 800]):
        row = subset[subset["radius_m"].eq(radius)].iloc[0]
        ax.errorbar(row["beta"], y, xerr=1.96 * row["se"], fmt="o", color=colors[radius], capsize=3)
    ax.axvline(0, color=PALETTE["grey"], linewidth=0.8)
    ax.set_yticks([0, 1, 2], ["300 m", "500 m", "800 m"] if ax is axes[0] else ["", "", ""])
    ax.set_title(outcome_label, fontsize=10)
    ax.set_xlabel("Standardized shock effect (95% CI)")
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("H1a Buffer Sensitivity", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(FIG_DIR / "fig_02_buffer_sensitivity.png", bbox_inches="tight")
plt.show()
display(buffer_results.round(4))

## 6. Alternative Commuter-Shock Definitions

Monday and Friday deficits may represent different hybrid-working behaviours. Year-specific shocks also show whether the result is driven by a single post-pandemic year. Every continuous exposure is standardised before estimation, so the coefficients are comparable in scale.

In [ ]:
alternative_labels = {
    "shock_cumulative": "Cumulative M+F",
    "shock_monday": "Monday only",
    "shock_friday": "Friday only",
    "shock_2023": "2023 M+F",
    "shock_2024": "2024 M+F",
    "shock_2025": "2025 M+F",
}
alternative_rows = []
panel_500 = panels[500]
for shock_column, shock_label in alternative_labels.items():
    for outcome_label, (outcome, baseline) in outcome_specs.items():
        result = fit_destination(panel_500, outcome, baseline, shock_column, standardize=True)
        if result:
            result.update({"shock_definition": shock_label, "outcome": outcome_label})
            alternative_rows.append(result)
alternative_results = pd.DataFrame(alternative_rows)
alternative_results.to_csv(OUT_DIR / "shock_definition_sensitivity_models.csv", index=False)

beta_matrix = alternative_results.pivot(index="outcome", columns="shock_definition", values="beta").reindex(index=outcome_specs, columns=alternative_labels.values())
p_matrix = alternative_results.pivot(index="outcome", columns="shock_definition", values="p_value").reindex(index=outcome_specs, columns=alternative_labels.values())
vmax = np.nanmax(np.abs(beta_matrix.values))
fig, ax = plt.subplots(figsize=(9.2, 3.6))
image = ax.imshow(beta_matrix.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(np.arange(len(beta_matrix.columns)), beta_matrix.columns, rotation=25, ha="right")
ax.set_yticks(np.arange(len(beta_matrix.index)), beta_matrix.index)
for i in range(beta_matrix.shape[0]):
    for j in range(beta_matrix.shape[1]):
        ax.text(j, i, f"{beta_matrix.iloc[i, j]:.3f}\np={p_matrix.iloc[i, j]:.3f}", ha="center", va="center", fontsize=7)
ax.set_title("H1a Sensitivity to the Commuter-Shock Definition", loc="left")
cbar = fig.colorbar(image, ax=ax, shrink=0.72)
cbar.set_label("Standardized shock coefficient")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_03_shock_definition_sensitivity.png", bbox_inches="tight")
plt.show()

## 7. Leave-One-MSOA-Out Influence Analysis

Each model is re-estimated after removing one workplace MSOA. If the coefficient changes sign repeatedly or moves far from the full-sample estimate, the result is sensitive to particular places and should not be presented as a stable London-wide relationship.

In [ ]:
influence_rows = []
for outcome_label, (outcome, baseline) in outcome_specs.items():
    full = fit_destination(panel_500, outcome, baseline, "shock_cumulative", standardize=True)
    for omitted in sorted(panel_500["MSOA21CD"].unique()):
        reduced = panel_500[panel_500["MSOA21CD"].ne(omitted)]
        result = fit_destination(reduced, outcome, baseline, "shock_cumulative", standardize=True)
        if result:
            influence_rows.append({
                "outcome": outcome_label, "omitted_msoa": omitted,
                "full_beta": full["beta"], "loo_beta": result["beta"],
                "loo_p": result["p_value"], "absolute_shift": abs(result["beta"] - full["beta"]),
            })
influence = pd.DataFrame(influence_rows)
influence.to_csv(OUT_DIR / "leave_one_msoa_out_models.csv", index=False)
influence_summary = influence.groupby("outcome", as_index=False).agg(
    full_beta=("full_beta", "first"), min_loo_beta=("loo_beta", "min"), max_loo_beta=("loo_beta", "max"),
    max_absolute_shift=("absolute_shift", "max"),
)
influence_summary["share_same_sign"] = influence_summary["outcome"].map(
    lambda label: np.mean(
        np.sign(influence.loc[influence["outcome"].eq(label), "loo_beta"])
        == np.sign(influence.loc[influence["outcome"].eq(label), "full_beta"].iloc[0])
    )
)
influence_summary.to_csv(OUT_DIR / "leave_one_msoa_out_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8.3, 3.5))
for y, row in influence_summary.iterrows():
    ax.hlines(y, row["min_loo_beta"], row["max_loo_beta"], color=PALETTE["blue"], linewidth=5, alpha=0.55)
    ax.plot(row["full_beta"], y, "o", color=PALETTE["orange_dark"], markersize=7)
ax.axvline(0, color=PALETTE["grey"], linewidth=0.8)
ax.set_yticks(np.arange(len(influence_summary)), influence_summary["outcome"])
ax.set_xlabel("Standardized shock coefficient")
ax.set_title("Leave-One-MSOA-Out Stability", loc="left")
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="none", markerfacecolor=PALETTE["orange_dark"], markeredgecolor=PALETTE["orange_dark"], label="Full estimate"),
    Line2D([0], [0], color=PALETTE["blue"], linewidth=5, alpha=0.55, label="Leave-one-out range"),
], frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_04_leave_one_msoa_out.png", bbox_inches="tight")
plt.show()
display(influence_summary.round(4))

## 8. Residual Moran's I

Moran's I tests whether model residuals remain geographically clustered. Because the selected workplace MSOAs are not all contiguous, four-nearest-neighbour weights are used. A significant result would indicate that nearby areas still share unexplained outcomes, motivating a spatial-error or spatial-lag sensitivity model.

In [ ]:
def morans_i_knn(values, coordinates, k=4, permutations=999, seed=42):
    values = np.asarray(values, dtype=float)
    coordinates = np.asarray(coordinates, dtype=float)
    n = len(values)
    distances = distance_matrix(coordinates, coordinates)
    np.fill_diagonal(distances, np.inf)
    neighbours = np.argsort(distances, axis=1)[:, :min(k, n - 1)]
    weights = np.zeros((n, n), dtype=float)
    for i, js in enumerate(neighbours):
        weights[i, js] = 1.0
    weights = np.maximum(weights, weights.T)
    row_sums = weights.sum(axis=1, keepdims=True)
    weights = np.divide(weights, row_sums, out=np.zeros_like(weights), where=row_sums > 0)
    centred = values - values.mean()
    denominator = np.sum(centred ** 2)

    def statistic(z):
        return (n / weights.sum()) * np.sum(weights * np.outer(z, z)) / np.sum(z ** 2)

    observed = statistic(centred)
    expected = -1 / (n - 1)
    rng = np.random.default_rng(seed)
    simulated = np.array([statistic(rng.permutation(centred)) for _ in range(permutations)])
    p_value = (np.sum(np.abs(simulated - expected) >= abs(observed - expected)) + 1) / (permutations + 1)
    return observed, expected, p_value

geometry = msoa[["MSOA21CD", "geometry"]].copy()
geometry["centroid_x"] = geometry.geometry.centroid.x
geometry["centroid_y"] = geometry.geometry.centroid.y
moran_rows = []
for outcome_label, (outcome, baseline) in outcome_specs.items():
    fitted = fit_destination(panel_500, outcome, baseline, "shock_cumulative", standardize=True, return_residuals=True)
    residual = fitted["residuals"].groupby("MSOA21CD", as_index=False)["residual"].mean().merge(
        geometry[["MSOA21CD", "centroid_x", "centroid_y"]], on="MSOA21CD", how="inner"
    )
    observed, expected, p_value = morans_i_knn(
        residual["residual"], residual[["centroid_x", "centroid_y"]], k=4
    )
    moran_rows.append({
        "outcome": outcome_label, "n_msoas": len(residual), "moran_i": observed,
        "expected_i": expected, "permutation_p": p_value, "weights": "4-nearest neighbours",
    })
moran_results = pd.DataFrame(moran_rows)
moran_results.to_csv(OUT_DIR / "residual_morans_i.csv", index=False)

display_table = moran_results.copy()
for column in ["moran_i", "expected_i", "permutation_p"]:
    display_table[column] = display_table[column].map(lambda value: f"{value:.3f}")
display_table = display_table.rename(columns={
    "outcome": "Outcome", "n_msoas": "MSOAs", "moran_i": "Moran's I",
    "expected_i": "Expected I", "permutation_p": "Permutation p", "weights": "Weights",
})
fig, ax = plt.subplots(figsize=(9.0, 2.5))
ax.axis("off")
ax.set_title("Residual Spatial Autocorrelation after the Main H1a Model", loc="left", fontsize=12)
table = ax.table(cellText=display_table.values, colLabels=display_table.columns, loc="center", cellLoc="left", colLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.45)
for (row, column), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d0d0")
    cell.set_linewidth(0.35)
    if row == 0:
        cell.set_facecolor("#e9edf2")
        cell.set_text_props(weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f8f9fb")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_05_residual_morans_i.png", bbox_inches="tight")
plt.show()
display(moran_results.round(4))

## 9. Automated Interpretation Summary

In [ ]:
interpretation = []
for _, row in power_table.iterrows():
    interpretation.append({
        "test": "Precision", "outcome": row["outcome_label"],
        "finding": f"|estimate| is {row['estimate_to_mde80']:.2f} of the estimated 80% MDE.",
        "implication": "Non-significance is compatible with limited precision." if row["estimate_to_mde80"] < 1 else "The estimated effect is at least as large as the approximate 80% MDE.",
    })
for outcome_label in outcome_specs:
    subset = continuous[continuous["outcome"].eq(outcome_label)]
    signs = np.sign(subset["beta"])
    interpretation.append({
        "test": "Buffer", "outcome": outcome_label,
        "finding": f"Coefficient direction is {'stable' if signs.nunique() == 1 else 'not stable'} across 300/500/800 m.",
        "implication": "The result is not driven by the 500 m rule alone." if signs.nunique() == 1 else "Spatial inclusion materially affects the estimated direction.",
    })
for _, row in influence_summary.iterrows():
    interpretation.append({
        "test": "Influence", "outcome": row["outcome"],
        "finding": f"{row['share_same_sign']:.0%} of leave-one-out estimates retain the full-sample sign.",
        "implication": "Direction is reasonably stable to individual MSOAs." if row["share_same_sign"] >= 0.9 else "Individual MSOAs materially affect the estimated direction.",
    })
for _, row in moran_results.iterrows():
    interpretation.append({
        "test": "Spatial residuals", "outcome": row["outcome"],
        "finding": f"Moran's I={row['moran_i']:.3f}, permutation p={row['permutation_p']:.3f}.",
        "implication": "Add a spatial-model sensitivity specification." if row["permutation_p"] < 0.05 else "No strong residual spatial autocorrelation is detected.",
    })
interpretation = pd.DataFrame(interpretation)
interpretation.to_csv(OUT_DIR / "robustness_interpretation_summary.csv", index=False)
display(interpretation)

## 10. Use in the Dissertation

- **Main methodology:** state that the 500 m cumulative Monday-Friday shock is pre-specified, while alternative buffers and shock definitions assess robustness.
- **Main results:** report the Notebook 07 coefficient figure and summarise whether the directions remain stable here.
- **Spatial model:** add a spatial-error or spatial-lag sensitivity model only when residual Moran's I is statistically meaningful and substantively non-trivial.
- **Appendix:** place the MDE, alternative-shock heatmap and leave-one-MSOA-out figure here unless one changes the interpretation of H1a.
- **Interpretive rule:** a non-significant coefficient is not converted into evidence of no effect. Discuss its confidence interval, detectable effect and stability.

Historical Green Street POI data will later add better-aligned outcomes such as openings, closures and tenant replacement. It may improve measurement of the mechanism, but it will not automatically increase the number of independent workplace exposure units.